In [1]:
import pandas as pd
from statsmodels.formula.api import ols
import statsmodels.api as sm
import math
import numpy as np
# from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.metrics import (
    mean_absolute_error, 
    median_absolute_error, 
    mean_squared_error, 
    mean_squared_log_error, 
    r2_score
)

In [2]:
# Đọc dữ liệu
df = pd.read_excel("data-for-regression-excel.xlsx", sheet_name="data")
df.columns = [col.replace(" ", "") for col in df.columns]

# Tách dữ liệu
X = df[["Area", "AccessRoad", "Bedrooms"]].values.tolist()
y_true = df["Price"].values
m = len(X)

In [3]:
# Dự đoán mô hình tuyến tính
def predict_linear(X, w):
    return [w[0] + w[1]*x[0] + w[2]*x[1] + w[3]*x[2] for x in X]

# Dự đoán mô hình phi tuyến
def predict_nonlinear(X, w):
    return [w[0]*math.log(x[0] + 1) + w[1]*x[1] + w[2]*x[2] + w[3] for x in X]

In [4]:
# Đánh giá mô hình
def evaluate(y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    mse = mean_squared_error(y_true, y_pred)
    rmse = math.sqrt(mse)
    r2 = r2_score(y_true, y_pred)
    return mae, mse, rmse, r2

In [ ]:
# Trọng số từ mô hình
w_linear = [2.2611, 0.0040, 0.0477, 0.7299]
# w1 = 0.6062409865804708, w2 = 0.041615163097700435, w3 = 0.716944620951016, w4 = 0.13858223174474435
w_nonlinear = [0.6062409865804708, 0.04161563097700435, 0.716944620951016, 0.13858223174474435]

In [6]:
# Tính chỉ số
results = {
    "Mô hình": [],
    "MAE": [],
    "MSE": [],
    "RMSE": [],
    "R²": []
}

for name, y_pred in [
    ("Tuyến tính L(X)", predict_linear(X, w_linear)),
    ("Phi tuyến M(X)", predict_nonlinear(X, w_nonlinear))
]:
    mae, mse, rmse, r2 = evaluate(y_true, y_pred)
    results["Mô hình"].append(name)
    results["MAE"].append(mae)
    results["MSE"].append(mse)
    results["RMSE"].append(rmse)
    results["R²"].append(r2)

results_df = pd.DataFrame(results)
display(results_df)

,Mô hình,MAE,MSE,RMSE,R²
0,Tuyến tính L(X),1.622275,3.954440,1.988577,0.212286
1,Phi tuyến M(X),1.609366,3.907577,1.976759,0.221621


In [7]:
## Detail
# Define extra metrics
def smape(y_true, y_pred):
    numerator = np.abs(y_true - y_pred)
    denominator = (np.abs(y_true) + np.abs(y_pred)) / 2
    smape_vals = numerator / np.where(denominator == 0, 1, denominator)
    return 100 * np.mean(smape_vals)

def mape(y_true, y_pred):
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    non_zero = y_true != 0
    return 100 * np.mean(np.abs((y_true[non_zero] - y_pred[non_zero]) / y_true[non_zero]))

def safe_rmsle(y_true, y_pred):
    y_true = np.where(np.array(y_true) < 0, 0, y_true)
    y_pred = np.where(np.array(y_pred) < 0, 0, y_pred)
    return np.sqrt(mean_squared_log_error(y_true + 1e-9, y_pred + 1e-9))

In [8]:
# Evaluate models
extended_results = {
    "Mô hình": [],
    "MAE": [],
    "MedAE": [],
    "MSE": [],
    "RMSE": [],
    "RMSLE": [],
    "MAPE (%)": [],
    "SMAPE (%)": [],
    "R²": []
}

for name, y_pred in [
    ("Tuyến tính L(X)", predict_linear(X, w_linear)),
    ("Phi tuyến M(X)", predict_nonlinear(X, w_nonlinear))
]:
    mae = mean_absolute_error(y_true, y_pred)
    medae = median_absolute_error(y_true, y_pred)
    mse = mean_squared_error(y_true, y_pred)
    rmse = math.sqrt(mse)
    rmsle = safe_rmsle(y_true, y_pred)
    mape_val = mape(y_true, y_pred)
    smape_val = smape(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)

    extended_results["Mô hình"].append(name)
    extended_results["MAE"].append(mae)
    extended_results["MedAE"].append(medae)
    extended_results["MSE"].append(mse)
    extended_results["RMSE"].append(rmse)
    extended_results["RMSLE"].append(rmsle)
    extended_results["MAPE (%)"].append(mape_val)
    extended_results["SMAPE (%)"].append(smape_val)
    extended_results["R²"].append(r2)

extended_df = pd.DataFrame(extended_results)

from IPython.display import display

display(extended_df)

# import ace_tools as tools
# tools.display_dataframe_to_user(name="Chỉ số đánh giá mô hình hồi quy", dataframe=extended_df)

,Mô hình,MAE,MedAE,MSE,RMSE,RMSLE,MAPE (%),SMAPE (%),R²
0,Tuyến tính L(X),1.622275,1.441300,3.954440,1.988577,0.335599,39.523806,31.874905,0.212286
1,Phi tuyến M(X),1.609366,1.435817,3.907577,1.976759,0.334283,39.314822,31.630875,0.221621
